In [2]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
sys.path.append(str(Path("..")))
from src.dataset import train_data, train_loader, val_loader, test_loader

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([3, 6, 0, 3, 2, 7, 5, 1, 5, 6])


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


# *DropOut 0.3*

In [4]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x


num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []

for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 1.5550 | Train Acc: 0.4512 | Val Loss: 0.8877 | Val Acc: 0.6937 | Val Macro F1: 0.6709
F1_Score by each class
[0.72592593 0.66176471 0.43055556 0.59574468 0.72972973 0.81339713
 0.88659794 0.52380952]
Epoch [2/100] | Train Loss: 0.7319 | Train Acc: 0.7429 | Val Loss: 0.7071 | Val Acc: 0.7733 | Val Macro F1: 0.7674
F1_Score by each class
[0.79452055 0.85561497 0.64705882 0.65365854 0.79289941 0.8245614
 0.86705202 0.7037037 ]
Epoch [3/100] | Train Loss: 0.4097 | Train Acc: 0.8566 | Val Loss: 0.5644 | Val Acc: 0.8078 | Val Macro F1: 0.7718
F1_Score by each class
[0.79268293 0.8994709  0.70157068 0.65895954 0.83615819 0.87378641
 0.95744681 0.45454545]
Epoch [4/100] | Train Loss: 0.2204 | Train Acc: 0.9287 | Val Loss: 0.6124 | Val Acc: 0.7958 | Val Macro F1: 0.7858
F1_Score by each class
[0.8630137  0.88324873 0.67605634 0.65116279 0.825      0.81967213
 0.90196078 0.66666667]
Epoch [5/100] | Train Loss: 0.1259 | Train Acc: 0.9572 | Val Loss: 0.6254 | Val Acc: 

# *DropOut 0.5*

In [5]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x


num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []

for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 1.4929 | Train Acc: 0.4876 | Val Loss: 0.9065 | Val Acc: 0.6712 | Val Macro F1: 0.6135
F1_Score by each class
[0.66666667 0.69333333 0.51351351 0.61904762 0.63565891 0.71161049
 0.89130435 0.17647059]
Epoch [2/100] | Train Loss: 0.7571 | Train Acc: 0.7226 | Val Loss: 0.6896 | Val Acc: 0.7808 | Val Macro F1: 0.7539
F1_Score by each class
[0.73770492 0.8627451  0.66371681 0.61878453 0.78527607 0.89215686
 0.94736842 0.52380952]
Epoch [3/100] | Train Loss: 0.4848 | Train Acc: 0.8311 | Val Loss: 0.6387 | Val Acc: 0.7838 | Val Macro F1: 0.7423
F1_Score by each class
[0.78014184 0.8839779  0.71559633 0.65048544 0.73684211 0.85321101
 0.93939394 0.37837838]
Epoch [4/100] | Train Loss: 0.3320 | Train Acc: 0.8863 | Val Loss: 0.5439 | Val Acc: 0.8153 | Val Macro F1: 0.7992
F1_Score by each class
[0.81632653 0.84466019 0.67027027 0.66285714 0.87179487 0.89622642
 0.96446701 0.66666667]
Epoch [5/100] | Train Loss: 0.2590 | Train Acc: 0.9114 | Val Loss: 0.5628 | Val Acc: